In [ ]:
# Plot confidence distribution of top-1 predictions
top1_confidences = []
for t in tracks:
    if t.classification and t.classification.genres:
        top1_confidences.append(t.classification.genres[0][1])

if top1_confidences:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(top1_confidences, bins=20, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Top-1 Genre Confidence")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Distribution of Top-1 Genre Confidence")
    axes[0].axvline(np.mean(top1_confidences), color="red", linestyle="--", 
                     label=f"Mean: {np.mean(top1_confidences):.3f}")
    axes[0].legend()
    
    # Per-track confidence bar chart
    track_names = [t.title[:25] for t in tracks if t.classification and t.classification.genres]
    axes[1].barh(track_names, top1_confidences)
    axes[1].set_xlabel("Top-1 Confidence")
    axes[1].set_title("Per-Track Top-1 Genre Confidence")
    axes[1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nMean top-1 confidence: {np.mean(top1_confidences):.3f}")
    print(f"Min: {min(top1_confidences):.3f}, Max: {max(top1_confidences):.3f}")
else:
    print("No classification data available.")

## 4. Confidence Distribution

How confident is the classifier? Low confidence across the board suggests the model struggles with our music.

In [ ]:
# Aggregate DJ genre scores across all tracks
genre_totals = {}
for profile in dj_profiles.values():
    for g, s in profile.genres.items():
        genre_totals[g] = genre_totals.get(g, 0) + s

# Sort and plot
genre_df = pd.DataFrame([
    {"genre": g, "total_score": s, "category": get_genre_category(g)}
    for g, s in sorted(genre_totals.items(), key=lambda x: -x[1])
])

if len(genre_df) > 0:
    fig, ax = plt.subplots(figsize=(14, max(6, len(genre_df) * 0.35)))
    colors = sns.color_palette("husl", n_colors=len(genre_df["category"].unique()))
    cat_colors = {cat: colors[i] for i, cat in enumerate(genre_df["category"].unique())}
    
    bars = ax.barh(
        genre_df["genre"], genre_df["total_score"],
        color=[cat_colors[c] for c in genre_df["category"]]
    )
    ax.set_xlabel("Total Score (summed across all tracks)")
    ax.set_title("DJ Genre Distribution in Library")
    ax.invert_yaxis()
    
    # Legend for categories
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=cat_colors[c], label=c) for c in cat_colors]
    ax.legend(handles=legend_elements, loc="lower right")
    
    plt.tight_layout()
    plt.show()
else:
    print("No genre data available. Analyze tracks first.")

## 3. Genre Distribution Across Library

Visualize what genres are present in the track library.

In [ ]:
# Map each track to DJ genres
dj_profiles = {}
for t in tracks:
    if not t.classification or not t.classification.genres:
        continue
    # Use all genre predictions (not just top-5) for taxonomy mapping
    profile = map_to_dj_genres(t.classification.genres)
    dj_profiles[t.title] = profile
    
    print(f"\n{t.title}:")
    print(f"  Primary: {profile.primary_genre} ({get_genre_category(profile.primary_genre)})")
    for g, s in profile.top_genres[:5]:
        bar = "█" * int(s * 30)
        print(f"  {g:<30} {bar} {s:.3f}")

## 2. DJ Taxonomy Mapping

Map the raw Discogs-400 predictions to our DJ-friendly genre taxonomy.

In [ ]:
# Build a dataframe of all genre predictions
rows = []
for t in tracks:
    if not t.classification or not t.classification.genres:
        continue
    for genre, conf in t.classification.genres[:5]:
        rows.append({"track": t.title, "discogs_genre": genre, "confidence": conf})

df_genres = pd.DataFrame(rows)
print(f"Total predictions: {len(df_genres)}")

# Show top-5 genres per track
for t in tracks:
    if not t.classification or not t.classification.genres:
        continue
    print(f"\n{t.title}:")
    for g, c in t.classification.genres[:5]:
        bar = "█" * int(c * 40)
        print(f"  {g:<45} {bar} {c:.3f}")

## 1. Raw Discogs-400 Predictions

For each track, show the top-5 Discogs genre predictions with confidence scores.

In [ ]:
import sys
sys.path.insert(0, "../src")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from agent_dj.analyzer.track_store import TrackStore
from agent_dj.analyzer.genre_taxonomy import map_to_dj_genres, get_genre_category

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

# Load analyzed tracks
store = TrackStore("../tracks.db")
tracks = store.get_all()
print(f"Loaded {len(tracks)} tracks from database")

# Experiment 01: Genre Classification Evaluation

Evaluate Essentia's pre-trained Discogs-400 genre classifier + our DJ taxonomy mapping.

**Goals:**
- How accurate are genre predictions on our sample tracks?
- Does the DJ taxonomy mapping produce sensible results?
- Which genres are well-classified? Which are confused?

**Requirements:** Drop audio files into `../samples/` and run `agent-dj analyze ../samples/` first.